# Signer-honest ASL recognition

Runs the full pipeline on a Colab GPU. Set **Runtime -> Change runtime type -> T4 GPU** first.

The experiment logic lives in `backend/src/slr/experiment.py`, so these cells stay short
and the science stays reviewable in one place instead of scattered across notebook state.

Two questions get answered, in order.

**1. How much of the published accuracy is real?** Same backbone, same data, three protocols:

| | protocol | what it measures |
| --- | --- | --- |
| A | random split | what most papers and Kaggle kernels report |
| B | group split | sessions kept whole |
| C | cross-corpus | tested on a corpus shot by different people |

**2. Which architecture is actually best?** A search over the model zoo, ranked on
**validation** only, with the test split touched once by the winner. Two of the
candidates are plain CNNs, deliberately: if a CNN ties the transformers, that is the
finding.

## 1. Environment and code

In [ ]:
!nvidia-smi -L
!pip install -q timm kaggle
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import os, sys
from pathlib import Path

REPO = "https://github.com/AlgoArtist06/signer-honest-asl.git"
if not Path("signer-honest-asl").exists():
    !git clone -q $REPO
os.chdir("/content/signer-honest-asl")
sys.path.insert(0, "/content/signer-honest-asl/backend/src")

from slr import sources, data, leakage, model as M, train, evaluate, experiment
print("loaded slr from", Path(experiment.__file__).parent)

## 2. Kaggle credentials

[kaggle.com/settings](https://www.kaggle.com/settings) -> **API** -> **Create Legacy API Key**
downloads `kaggle.json`. Paste the two values below, or upload the file.

Nothing here is committed: `dataset/raw/` and credentials are gitignored.

In [ ]:
KAGGLE_USERNAME = ""   # e.g. "ashutosh839"
KAGGLE_KEY      = ""   # the 32-char key from kaggle.json

from pathlib import Path
if KAGGLE_USERNAME and KAGGLE_KEY:
    experiment.write_kaggle_credentials(KAGGLE_USERNAME, KAGGLE_KEY)
elif not (Path.home() / ".kaggle" / "kaggle.json").exists():
    from google.colab import files
    p = Path.home() / ".kaggle"; p.mkdir(parents=True, exist_ok=True)
    (p / "kaggle.json").write_bytes(files.upload()["kaggle.json"])
    (p / "kaggle.json").chmod(0o600)

## 3. Data

Downloads `asl_alphabet` (87k images, ~1.1 GB) and `asl_alphabet_test` (870 images shot
by a different person), builds the manifest, and recovers capture sessions by clustering
near-duplicates.

Session recovery is what makes protocol B possible at all: neither corpus ships signer
labels, so the sessions have to be inferred from the images themselves.

In [ ]:
sources.main(["list"])

In [ ]:
experiment.run("data", per_class=0)

## 4. Protocol A - the leaky baseline

A random split scatters each capture session across train and test, so near-identical
frames sit on both sides. The audit reports exactly how contaminated the result is.

`per_class` caps images per class by dropping **whole groups**, never individual frames,
so subsampling cannot itself create or hide a leak.

In [ ]:
PER_CLASS = 400      # whole groups only; set 0 to use all 87k images
PRESET    = "vit_small"
EPOCHS    = 4

experiment.run("A", per_class=PER_CLASS, preset=PRESET, epochs=EPOCHS)

## 5. Protocol B - honest in-corpus split

Same corpus, same backbone, same epochs. The only change is that recovered sessions are
dealt whole to exactly one of train / val / test.

In [ ]:
experiment.run("B", per_class=PER_CLASS, preset=PRESET, epochs=EPOCHS)

## 6. Protocol C - cross-corpus

Train on `asl_alphabet`, test on `asl_alphabet_test`: different person, different room,
different camera. Validation is carved out of the training corpus, so the test corpus is
untouched until the final number.

This is signer-disjoint by construction, which matters because session recovery alone
cannot guarantee signer disjointness - see `leakage.recover_groups` for the measurement
that establishes this.

In [ ]:
experiment.run("C", per_class=PER_CLASS, preset=PRESET, epochs=EPOCHS)

## 7. The headline table

In [ ]:
experiment.summary()

## 8. Architecture search

Every backbone the reading list pointed at, ranked on validation macro-F1.
Only the winner is run against the test split.

The sweep is resumable: `runs/C_arch_leaderboard.json` is rewritten after each
candidate and re-read on restart, so a dropped session costs the model in flight,
not the sweep. Re-run this cell after a reconnect and it picks up where it stopped
(re-run section 3 first, since the VM's disk is wiped with it).

In [ ]:
print({k: v for k, v in M.SWEEPS.items()})

In [ ]:
SEARCH = "all"      # "quick" | "scale" | "arch" | "mobile" | "all"

# Cheapest first. A full-zoo sweep is longer than a free Colab session, and the
# board is written after every candidate, so ordering by cost means a
# disconnect costs the three 448/518px monsters rather than fifteen models.
ORDER = sorted(M.SWEEPS[SEARCH],
               key=lambda k: (M.PRESETS[k].img_size, M.PRESETS[k].approx_params_m))
print(len(ORDER), "candidates:", ORDER)

arch = experiment.run("arch", per_class=PER_CLASS, presets=ORDER, epochs=EPOCHS)
BEST = arch["winner"]
print("winner:", BEST, "| test:", arch["winner_test"])

### Reading the leaderboard

Three things to check before quoting the top row:

- **Did a CNN win?** `convnext_base` and `effnetv2_s` are controls. If one leads,
  attention is not what carried this task.
- **Did the language- or self-supervised models beat the ImageNet ones?** `siglip_base`,
  `clip_base` and `dinov2_base` saw far more visual variety in pretraining. If they lead
  on the cross-corpus test specifically, the finding is that pretraining diversity, not
  architecture, buys robustness to a new signer.
- **What did the parameters cost?** Compare `val_f1` against `secs_per_epoch`. The
  deployment pick is the knee of that curve, not the top of the table.

In [ ]:
import pandas as pd

board = pd.DataFrame(arch["leaderboard"])
board["f1_per_Mparam"] = (board.val_f1 / board.params_m * 100).round(3)
board

## 9. Calibration and the abstention threshold

Temperature is fitted on validation only. The risk-coverage table is where the app's
confidence threshold comes from: at a given coverage, this is the measured error rate
among the answers the model is willing to give.

In [ ]:
cal = experiment.run("cal", per_class=PER_CLASS, winner=BEST)
cal["test"]["risk_coverage"]

## 10. Take the model home

Downloads the checkpoint plus its calibration report. Keep them together: the API reads
`eval.json` from beside the `.pt` to get the temperature and threshold.

In [ ]:
import shutil, json
from pathlib import Path
from google.colab import files

ck = Path(cal["checkpoint"]).parent
shutil.make_archive("slr_model", "zip", ck)
files.download("slr_model.zip")
print(json.dumps(experiment.load()["stages"], indent=2)[:2000])